# Lesson 41: Machine translation and sequence-to-sequence demonstration

This notebook demonstrates sequence preprocessing and encoder-decoder architecture for machine translation.

**1. Sequence preprocessing**
- 1.1. Tokenization and vocabulary
- 1.2. Padding and sequencing

**2. Encoder-decoder architecture**
- 2.1. Model architecture
- 2.2. Training
- 2.3. Inference


## Notebook set up

### Imports

In [ ]:
import os
import numpy as np
import pandas as pd

# Set environment variables for TensorFlow (must be done before import)
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.environ['TF_USE_CUDNN_RNN'] = '0'

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

### Configuration

In [ ]:
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

### Parallel corpus

A small English-French parallel corpus for demonstration.

In [ ]:
# Simple English-French parallel corpus
english_sentences = [
    'hello',
    'how are you',
    'thank you',
    'good morning',
    'good night',
    'i love you',
    'goodbye',
    'yes',
    'no',
    'please'
]

french_sentences = [
    'bonjour',
    'comment allez vous',
    'merci',
    'bonjour',
    'bonne nuit',
    'je vous aime',
    'au revoir',
    'oui',
    'non',
    'sil vous plait'
]

# Add start and end tokens to target sentences
french_input = ['startseq ' + s for s in french_sentences]
french_output = [s + ' endseq' for s in french_sentences]

print(f'Parallel corpus size: {len(english_sentences)} sentence pairs')
pd.DataFrame({'English': english_sentences, 'French': french_sentences})

## 1. Sequence preprocessing

### 1.1. Tokenization and vocabulary

Convert text to integer sequences using a vocabulary index.

Keras [Tokenizer](https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/text/Tokenizer) documentation

In [14]:
# Create tokenizers for source and target languages
eng_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(english_sentences)
eng_vocab_size = len(eng_tokenizer.word_index) + 1

fra_tokenizer = Tokenizer()
fra_tokenizer.fit_on_texts(french_input + french_output)
fra_vocab_size = len(fra_tokenizer.word_index) + 1

print(f'English vocabulary size: {eng_vocab_size}')
print(f'French vocabulary size: {fra_vocab_size}')
print(f'\nEnglish vocabulary: {eng_tokenizer.word_index}')
print(f'\nFrench vocabulary: {fra_tokenizer.word_index}')

English vocabulary size: 15
French vocabulary size: 18

English vocabulary: {'you': 1, 'good': 2, 'hello': 3, 'how': 4, 'are': 5, 'thank': 6, 'morning': 7, 'night': 8, 'i': 9, 'love': 10, 'goodbye': 11, 'yes': 12, 'no': 13, 'please': 14}

French vocabulary: {'start': 1, 'end': 2, 'vous': 3, 'bonjour': 4, 'comment': 5, 'allez': 6, 'merci': 7, 'bonne': 8, 'nuit': 9, 'je': 10, 'aime': 11, 'au': 12, 'revoir': 13, 'oui': 14, 'non': 15, 'sil': 16, 'plait': 17}


### 1.2. Padding and sequencing

Sequences must have the same length for batch processing. Padding adds zeros to shorter sequences.

Keras [pad_sequences](https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/sequence/pad_sequences) documentation

In [ ]:
# Convert to sequences
eng_sequences = eng_tokenizer.texts_to_sequences(english_sentences)
fra_input_sequences = fra_tokenizer.texts_to_sequences(french_input)
fra_output_sequences = fra_tokenizer.texts_to_sequences(french_output)

# Find max sequence lengths
max_eng_len = max(len(seq) for seq in eng_sequences)
max_fra_len = max(len(seq) for seq in fra_input_sequences)

# Pad sequences
encoder_input = pad_sequences(eng_sequences, maxlen=max_eng_len, padding='post')
decoder_input = pad_sequences(fra_input_sequences, maxlen=max_fra_len, padding='post')
decoder_output = pad_sequences(fra_output_sequences, maxlen=max_fra_len, padding='post')

print(f'Encoder input shape: {encoder_input.shape}')
print(f'Decoder input shape: {decoder_input.shape}')
print(f'\nExample (first sentence):')
print(f'  English: {english_sentences[0]} -> {encoder_input[0]}')
print(f'  French input: {french_input[0]} -> {decoder_input[0]}')
print(f'  French output: {french_output[0]} -> {decoder_output[0]}')

## 2. Encoder-decoder architecture

### 2.1. Model architecture (LSTM)

The encoder processes the input sequence and produces a context vector. The decoder uses this context to generate the output sequence.

```text
                    ENCODER                                      DECODER
                                                                                       
  Input sequence                                   Target sequence (shifted)          
       │                                                   │                          
       ▼                                                   ▼                          
┌─────────────┐                                     ┌─────────────┐                   
│  Embedding  │                                     │  Embedding  │                   
└──────┬──────┘                                     └──────┬──────┘                   
       │                                                   │                          
       ▼                                                   ▼                          
┌─────────────┐    Context vector (h, c)            ┌─────────────┐                   
│    LSTM     │ ─────────────────────────────────►  │    LSTM     │                   
└─────────────┘                                     └──────┬──────┘                   
                                                           │                          
                                                           ▼                          
                                                    ┌─────────────┐                   
                                                    │   Dense     │                   
                                                    │  (softmax)  │                   
                                                    └──────┬──────┘                   
                                                           │                          
                                                           ▼                          
                                                    Output sequence                    
```

**Context vector (h, c):** The LSTM produces two state vectors that form the context:
- **h (hidden state):** The current output representation encoding what the LSTM has "decided" to output
- **c (cell state):** The long-term memory that carries information across the sequence

**Target sequence shifting:** During training, we use "teacher forcing" where the decoder input is shifted by one position relative to the expected output:
- Decoder input: `startseq bonjour` (what the model sees)
- Decoder target: `bonjour endseq` (what the model learns to predict)

This teaches the model to predict the next token given all previous tokens.

Keras [LSTM](https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM) documentation

In [ ]:
# Model parameters
embedding_dim = 64
hidden_units = 128

# Encoder (recurrent_dropout forces non-CuDNN implementation)
encoder_inputs = Input(shape=(max_eng_len,), name='encoder_input')
encoder_embedding = Embedding(eng_vocab_size, embedding_dim, name='encoder_embedding')(encoder_inputs)

encoder_lstm, state_h, state_c = LSTM(
    hidden_units, return_state=True, recurrent_dropout=1e-5, name='encoder_lstm'
)(encoder_embedding)

encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(max_fra_len,), name='decoder_input')
decoder_embedding = Embedding(fra_vocab_size, embedding_dim, name='decoder_embedding')(decoder_inputs)

decoder_lstm = LSTM(
    hidden_units, return_sequences=True, return_state=True, recurrent_dropout=1e-5, name='decoder_lstm'
)

decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)
decoder_dense = Dense(fra_vocab_size, activation='softmax', name='decoder_output')
decoder_outputs = decoder_dense(decoder_outputs)

# Build model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.summary()

Model: "model_3"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 encoder_input (InputLayer)  [(None, 3)]                  0         []                            
                                                                                                  
 decoder_input (InputLayer)  [(None, 4)]                  0         []                            
                                                                                                  
 encoder_embedding (Embeddi  (None, 3, 64)                960       ['encoder_input[0][0]']       
 ng)                                                                                              
                                                                                                  
 decoder_embedding (Embeddi  (None, 4, 64)                1152      ['decoder_input[0][0]'] 

### 2.2. Training

In [11]:
# Prepare target data (add dimension for sparse categorical crossentropy)
decoder_target = np.expand_dims(decoder_output, -1)

# Train model
history = model.fit(
    [encoder_input, decoder_input],
    decoder_target,
    epochs=100,
    batch_size=5,
    verbose=0
)

print(f'Final training loss: {history.history["loss"][-1]:.4f}')
print(f'Final training accuracy: {history.history["accuracy"][-1]:.4f}')

Final training loss: 0.0875
Final training accuracy: 1.0000


### 2.3. Inference

For inference, we need separate encoder and decoder models to generate translations step by step.

In [ ]:
# Create inference encoder model
encoder_model = Model(encoder_inputs, encoder_states)

# Create inference decoder model
decoder_state_input_h = Input(shape=(hidden_units,))
decoder_state_input_c = Input(shape=(hidden_units,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    decoder_embedding,
    initial_state=decoder_states_inputs
)

decoder_states_inf = [state_h_inf, state_c_inf]
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf
)

### Translation process

The `translate` function generates output tokens one at a time:

1. Encode the input sentence to get the initial hidden states (h, c)
2. Start decoder with the `startseq` token
3. Loop: predict next token, append to output, feed prediction back as next input
4. Stop when `endseq` is predicted or max length reached

In [15]:
# Build reverse vocabulary for decoding
fra_index_to_word = {v: k for k, v in fra_tokenizer.word_index.items()}

# Get start/end token keys from vocabulary
start_token = 'startseq' if 'startseq' in fra_tokenizer.word_index else 'start'
end_token = 'endseq' if 'endseq' in fra_tokenizer.word_index else 'end'

def translate(input_sentence):

    # Encode input
    input_seq = eng_tokenizer.texts_to_sequences([input_sentence])
    input_seq = pad_sequences(input_seq, maxlen=max_eng_len, padding='post')

    # Get encoder states
    states = encoder_model.predict(input_seq, verbose=0)

    # Start with start token
    target_seq = np.zeros((1, max_fra_len))
    target_seq[0, 0] = fra_tokenizer.word_index[start_token]

    # Generate translation
    translation = []

    for i in range(max_fra_len - 1):

        output, h, c = decoder_model.predict([target_seq] + states, verbose=0)
        predicted_id = np.argmax(output[0, i, :])

        if predicted_id == 0 or fra_index_to_word.get(predicted_id) == end_token:
            break

        word = fra_index_to_word.get(predicted_id, '')

        if word and word != start_token:
            translation.append(word)

        target_seq[0, i + 1] = predicted_id
        states = [h, c]

    return ' '.join(translation)

# Test translations
print('Translation examples:')

for eng in english_sentences[:5]:
    fra = translate(eng)
    print(f'  {eng} -> {fra}')

Translation examples:
  hello -> bonjour
  how are you -> comment
  thank you -> merci
  good morning -> bonjour
  good night -> bonne
